In [ ]:
from pathlib import Path
import json
import textwrap
import gzip
DATA_PATH = Path("/home/dstartsev/dataset/HumanEval/human-eval/data") / "HumanEval.jsonl.gz"

tasks = []
with gzip.open(DATA_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        tasks.append(json.loads(line))

print("Total tasks:", len(tasks))
print("Example of fields:", tasks[0].keys())
print("The first task_id:", tasks[0]["task_id"])

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import re
from tqdm.auto import tqdm
import torch

In [ ]:
REPO_ID_DIFFU = "/home/dstartsev/models/DiffuCoder-7B-Instruct"  

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer_diffu = AutoTokenizer.from_pretrained(
    REPO_ID_DIFFU,
    trust_remote_code=True,
)

model_diffu = AutoModel.from_pretrained(
    REPO_ID_DIFFU,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

model_diffu.eval()
print("DiffuCoder-7B-Instruct loaded")

In [ ]:
TEMPERATURE = 0.1
TOP_P = 0.95
ALG_TEMP = 0.0  
EOS_PENALTY = 3.0
results = []

In [ ]:
def clean_and_check(code: str):
    code = textwrap.dedent(code).strip()
    lines = code.splitlines()
    start = None
    for i, line in enumerate(lines):
        if line.lstrip().startswith("def "):
            start = i
            break
    if start is not None:
        code = "\n".join(lines[start:])
    try:
        ast.parse(code)
        return code, True
    except SyntaxError:
        return code, False

In [ ]:
def make_humaneval_prompt_diffu(prompt_text: str):
    messages = [{"role": "user", "content": prompt_text.strip()}]
    return tokenizer_diffu.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True,
    )

In [ ]:
def extract_first_function(text: str) -> str:
    lines = text.splitlines()

    start = None
    for i, line in enumerate(lines):
        if line.lstrip().startswith("def "):
            start = i
            break
    if start is None:
        return text  

    func_lines = [lines[start]]
    base_indent = len(lines[start]) - len(lines[start].lstrip())


    for line in lines[start+1:]:

        if line.strip() == "":
            func_lines.append(line)
            continue

        indent = len(line) - len(line.lstrip())
        if indent > base_indent:
            func_lines.append(line)
        else:
            
            break

    return "\n".join(func_lines)

In [ ]:
def generate_with_diffucoder_params(prompt: str, steps: int, max_new: int):
    inputs = make_humaneval_prompt_diffu(prompt)
    input_ids = inputs["input_ids"].to(model_diffu.device)
    attention_mask = inputs["attention_mask"].to(model_diffu.device)

    with torch.no_grad():
        out = model_diffu.diffusion_generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new,
            steps=steps,
            temperature=TEMPERATURE,  
            top_p=TOP_P,               
            alg="entropy",
            alg_temp=ALG_TEMP,
            output_history=False,
            return_dict_in_generate=True,
        )

    input_len = input_ids.shape[1]
    gen_ids = out.sequences[0, input_len:]
    gen_text = tokenizer_diffu.decode(gen_ids, skip_special_tokens=True)

    if tokenizer_diffu.eos_token is not None:
        gen_text = gen_text.split(tokenizer_diffu.eos_token)[0]

    
    gen_text = extract_first_function(gen_text)

    code, _ = clean_and_check(gen_text)
    return code, gen_ids.shape[-1]

In [ ]:
from typing import Dict, Any
import signal

def run_tests_for_task(task: Dict[str, Any], gen_code: str) -> bool:
    prompt = task["prompt"]
    tests = task["test"]
    entry_point = task["entry_point"]
    full_src = prompt + "\n" + gen_code + "\n" + tests
    
    ns: Dict[str, Any] = {}
    try:
        exec(textwrap.dedent(full_src), ns)
    except Exception as e:
        print(f"[EXEC ERROR] {task['task_id']}: {e}")
        return False
    
    if "check" not in ns or entry_point not in ns:
        print(f"[NS MISSING] {task['task_id']}: 'check' or '{entry_point}' not in namespace")
        return False
    
    try:
        signal.alarm(3)
        ns["check"](ns[entry_point])
        signal.alarm(0)
        return True
    except Exception as e:
        signal.alarm(0)
        print(f"[CHECK ERROR] {task['task_id']}: {e}")
        return False

In [ ]:
import ast
from tqdm.auto import tqdm
import pandas as pd
import time
import json

steps_list = [16, 32, 64, 128, 256, 512]   
max_new_list = [256, 512]


checkpoint_path = "humaneval_diffucoder_grid_checkpoint.csv"
per_sample_path = "humaneval_diffucoder_samples.jsonl"

all_rows = []

for steps in steps_list:
    for max_new in max_new_list:
        print(f"\n=== [DiffuCoder] STEPS = {steps}, MAX_NEW = {max_new} ===")

        n_tasks = 0
        n_pass = 0
        n_ast_ok = 0
        total_len_tokens = 0
        n_trunc = 0

        t_start = time.time()

        for task in tqdm(tasks):   
            n_tasks += 1
            task_id = task["task_id"]
            prompt = task["prompt"]

            gen_code, gen_len_tokens = generate_with_diffucoder_params(
                prompt, steps, max_new
            )

            try:
                ast.parse(gen_code)
                ast_ok = True
            except SyntaxError as e:
                print(f"[AST ERROR] {task_id} @ steps={steps}, max_new={max_new}: {e}")
                ast_ok = False

            if ast_ok:
                n_ast_ok += 1
            if gen_len_tokens >= max_new:
                n_trunc += 1

            total_len_tokens += gen_len_tokens

            passed = run_tests_for_task(task, gen_code) if ast_ok else False
            if passed:
                n_pass += 1

            print(f"{task_id}: {'Correct' if passed else 'Incorrect'}")

            with open(per_sample_path, "a", encoding="utf-8") as f:
                f.write(json.dumps({
                    "model": "diffucoder-7b-instruct",
                    "steps": steps,
                    "max_new_tokens": max_new,
                    "task_id": task_id,
                    "ast_ok": ast_ok,
                    "passed": passed,
                    "gen_len_tokens": gen_len_tokens,
                }, ensure_ascii=False) + "\n")

        t_end = time.time()
        wall_time = t_end - t_start

        pass_at_1 = n_pass / n_tasks if n_tasks > 0 else 0.0
        ast_validity = n_ast_ok / n_tasks if n_tasks > 0 else 0.0
        trunc_rate = n_trunc / n_tasks if n_tasks > 0 else 0.0
        avg_len_tokens = total_len_tokens / n_tasks if n_tasks > 0 else 0.0
        avg_time_per_task = wall_time / n_tasks if n_tasks > 0 else 0.0

        print(
            f"RESULTS [DiffuCoder] steps={steps}, max_new={max_new}: "
            f"Pass@1={pass_at_1:.3f}, AST-validity={ast_validity:.3f}, "
            f"Trunc={trunc_rate:.3f}, avg_len={avg_len_tokens:.1f}, "
            f"time/task={avg_time_per_task:.2f}s"
        )

        all_rows.append({
            "model": "diffucoder-7b-instruct",
            "steps": steps,
            "max_new_tokens": max_new,
            "num_tasks": n_tasks,
            "pass_at_1": pass_at_1,
            "ast_validity": ast_validity,
            "truncation_rate": trunc_rate,
            "avg_len_tokens": avg_len_tokens,
            "avg_time_per_task_sec": avg_time_per_task,
            "total_time_sec": wall_time,
        })

        pd.DataFrame(all_rows).to_csv(checkpoint_path, index=False)
        print(f"Checkpoint saved to {checkpoint_path}")


pd.DataFrame(all_rows).to_csv("humaneval_diffucoder_grid_final.csv", index=False)
print("Final saved to humaneval_diffucoder_grid_final.csv")